# SEM defect inspection — Colab launcher

Runs the benchmark configs on a Colab GPU (T4 / L4 / A100); the full local plan is `plan()` in `scripts/run_campaign.py`.

* Code: cloned (or uploaded) to `/content/SEM_Defect_Inspection`.
* **Everything durable lives on Google Drive** (`/content/drive/MyDrive/semdefect`): dataset, caches,
  `outputs/` (checkpoints) and `mlruns/` (MLflow file store). Colab has no background execution, so a
  disconnect only loses the minutes since the last checkpoint: every training script saves
  `last.ckpt` at least every 15 min (`checkpoint_minutes`) and resumes automatically (`resume: true`).
* Re-run the notebook top to bottom after a disconnect; finished runs are skipped, unfinished ones resume.

## 1. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/semdefect')
DRIVE.mkdir(parents=True, exist_ok=True)
REPO = pathlib.Path('/content/SEM_Defect_Inspection')
REPO_URL = ''  # e.g. 'https://github.com/<you>/SEM_Defect_Inspection.git' (leave empty to use a zip on Drive)

if not REPO.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    else:  # fallback: a zip of the repo uploaded to Drive (without data/ and .venv/)
        subprocess.run(['unzip', '-q', str(DRIVE / 'SEM_Defect_Inspection.zip'), '-d', '/content'], check=True)
%cd /content/SEM_Defect_Inspection

## 2. Install (Colab already ships CUDA PyTorch)

In [ ]:
!pip -q install timm mlflow pyyaml scikit-learn scipy pandas
!pip -q install -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Persist outputs / MLflow on Drive; data + caches on local SSD

In [ ]:
# Durable on Drive: outputs/ (checkpoints, metrics) and mlruns/ (MLflow file store).
for sub in ['outputs', 'mlruns']:
    target = DRIVE / sub
    target.mkdir(parents=True, exist_ok=True)
    link = REPO / sub
    if link.exists() and not link.is_symlink():
        subprocess.run(['rm', '-rf', str(link)])
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)

# Data + memmap caches on the fast local SSD (memmaps over the Drive FUSE mount are slow);
# caches are persisted to Drive once and copied back each session (~3.6 GB, 1-2 min).
!python scripts/download_data.py          # 133 MB from Zenodo, md5-verified
!python -m semdefect.cli prepare --seeds 0 1 2
CACHE_DRIVE = DRIVE / 'cache_arrays'
if CACHE_DRIVE.exists():
    !mkdir -p data/cache/arrays && rsync -a {CACHE_DRIVE}/ data/cache/arrays/
!python scripts/build_cache.py            # no-op when the copied caches are complete
!mkdir -p {CACHE_DRIVE} && rsync -a data/cache/arrays/ {CACHE_DRIVE}/

## 4. Pick runs

Each entry is `(config, overrides)`. A run whose `outputs/<name>/test_metrics.json` exists is skipped.
Training runs resume from `outputs/<name>/checkpoints/last.ckpt`. The full experiment plan is
`plan()` in `scripts/run_campaign.py`.

In [ ]:
RUNS = [
    ('configs/classify/fewshot_dinov2_vits14.yaml', []),
    ('configs/classify/fewshot_dinov2_vitb14.yaml', []),
    ('configs/classify/resnet18_full.yaml', []),
    ('configs/segment/unet_r18.yaml', []),
    ('configs/segment/dino_seg_vits14.yaml', []),
    ('configs/anomaly/patchcore.yaml', ['robustness.enabled=true']),
    ('configs/anomaly/anomalydino.yaml', ['robustness.enabled=true']),
    ('configs/anomaly/efficientad.yaml', ['train.steps=20000', 'train.batch_size=8']),
    ('configs/anomaly/dinomaly.yaml', []),
]
# k-shot fine-tuning of the CNN baseline
RUNS += [('configs/classify/resnet18_fewshot.yaml', [f'fewshot.k={k}', f'fewshot.seed={s}', f'name=resnet18_k{k}_s{s}'])
         for k in (1, 5, 10, 20) for s in range(3)]

In [ ]:
import json, yaml, sys
sys.path.insert(0, 'src')
from semdefect.config import load_config

def done(cfg_path, overrides):
    cfg = load_config(cfg_path, overrides)
    out = REPO / 'outputs' / cfg['name']
    return any((out / f).exists() for f in ('test_metrics.json', 'fewshot_results.json'))

for cfg_path, ov in RUNS:
    if done(cfg_path, ov):
        print('skip (done):', cfg_path, ov); continue
    print('=== run', cfg_path, ov, flush=True)
    r = subprocess.run([sys.executable, '-m', 'semdefect.cli', 'run', cfg_path, 'data.num_workers=2', *ov])
    if r.returncode != 0:
        print('FAILED', cfg_path, ov)

## 5. Results

In [ ]:
!python scripts/collect_results.py
!mlflow ui --backend-store-uri ./mlruns --port 5000 &  # optional, use a Colab port tunnel to view